Silver layer: UKHSA COVID-19 cases, cleaned and typed

Reads every Bronze JSON page under the COVID-19 cases metric, explodes the
results array so each record is a row, casts types, filters bad rows,
dedupes, and writes a clean Delta table to Silver.

In [0]:
# Checking fieldnames

# File path to the bronze layer container
bronze_path = "abfss://bronze@ukhsadev2026.dfs.core.windows.net/ukhsa/COVID-19/COVID-19_cases_casesByDay/"

# Recursively find all of the json files from the bronze layer root down.
raw_df = spark.read.option("recursiveFileLookup", "true").json(bronze_path)

# Imports
from pyspark.sql.functions import explode, col

# Selecting results from the json, discarding the additional metadata
# The single results row is expanded (exploded) into a column (r) with each row being a different entry day
exploded_df = raw_df.select(explode(col("results")).alias("r"))
# Inspecting the fieldnames.
exploded_df.select("r.*").printSchema()

# Checking if there are unique combinations of stratum, sex and age.
# If there is more than one combination then a single date will have multiple entries instead of one.
# Eg. One for male and one for female or all together 
# If there is only one unique combination then we know that there should be one row per date per location
exploded_df.select("r.stratum", "r.sex", "r.age").distinct().show(50, truncate=False)

# Result is default, all, all so they are not separated by sex age or stratum.

In [0]:
# Imports
from pyspark.sql.functions import explode, col, to_date
from pyspark.sql.types import FloatType, IntegerType

# File paths for bronze layer (input) and silver layer (output)
bronze_path = "abfss://bronze@ukhsadev2026.dfs.core.windows.net/ukhsa/COVID-19/COVID-19_cases_casesByDay/"
silver_path = "abfss://silver@ukhsadev2026.dfs.core.windows.net/ukhsa/covid19_cases_by_day"

# Read every Bronze JSON page in one pass (recursively)
raw_df = spark.read.option("recursiveFileLookup", "true").json(bronze_path)

# Explode results so each record is its own row
exploded_df = raw_df.select(explode(col("results")).alias("r"))

# Flatten (pulling out nested data into flat columns)
typed_df = exploded_df.select(
    col("r.theme").alias("theme"),
    col("r.sub_theme").alias("sub_theme"),
    col("r.topic").alias("topic"),
    col("r.geography_type").alias("geography_type"),
    col("r.geography").alias("geography"),
    col("r.geography_code").alias("geography_code"),
    col("r.metric").alias("metric"),
    col("r.stratum").alias("stratum"),
    col("r.sex").alias("sex"),
    col("r.age").alias("age"),
    to_date(col("r.date")).alias("date"),
    col("r.metric_value").alias("metric_value"),
    col("r.in_reporting_delay_period").alias("in_reporting_delay_period"),
)

# Filter and data cleaning
# Remove incomplete rows and any rows in the delay window
clean_df = typed_df.filter(
    col("date").isNotNull()
    & col("metric_value").isNotNull()
    & (col("in_reporting_delay_period") == False)
)

# Create dataframe with only unique instances to make sure there are no repeats.
# Every row must be a unique combination of date, geography code (location), and metric (value - covid cases)
deduped_df = clean_df.dropDuplicates(
    ["geography_code", "metric", "date"]
)

# Create a delta table and save it to the silver layer external location
deduped_df.write.format("delta").mode("overwrite").save(silver_path)

print(f"Silver write complete: {deduped_df.count()} rows")